# 🎮 Video Game Analytics — SQL Setup
## PostgreSQL + pgAdmin4
**Database:** `videogame_analytics`


## Step 1 — Libraries & Connection

In [10]:
# !pip install psycopg2-binary sqlalchemy pandas

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

✅ Libraries loaded


In [11]:
from urllib.parse import quote_plus  # ← ye line add karo

# ── Apna password yahan update karo ──
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'videogame_analytics'
DB_USER = 'postgres'
DB_PASS = quote_plus('Rudra@2001')   # 👈 YAHAN APNA PASSWORD DAALO

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    echo=False
)

with engine.connect() as conn:
    row = conn.execute(text('SELECT current_database(), current_user')).fetchone()
    print(f'✅ Connected to database : {row[0]}')
    print(f'   User                  : {row[1]}')

✅ Connected to database : videogame_analytics
   User                  : postgres


## Step 2 — CSV Files Load Karo

In [12]:
games   = pd.read_csv('cleaned_data/cleaned_games.csv')
vgsales = pd.read_csv('cleaned_data/cleaned_vgsales.csv')
merged  = pd.read_csv('cleaned_data/merged_data.csv')

print(f'✅ games   loaded : {games.shape}')
print(f'✅ vgsales loaded : {vgsales.shape}')
print(f'✅ merged  loaded : {merged.shape}')

✅ games   loaded : (1099, 19)
✅ vgsales loaded : (16593, 15)
✅ merged  loaded : (1099, 27)


## Step 3 — Dimension Tables

> Schema pehle se pgAdmin4 mein run ho chuka hai ✅

In [16]:
# ── dim_genre ─────────────────────────────────────────
all_genres = set()
for g in games['Primary_Genre'].dropna():   all_genres.add(str(g).strip())
for g in vgsales['Genre'].dropna():         all_genres.add(str(g).strip())

pd.DataFrame({'genre_name': sorted(all_genres)}).to_sql(
    'dim_genre', engine, if_exists='append', index=False)
print(f'✅ dim_genre    : {len(all_genres)} rows')

# ── dim_platform ──────────────────────────────────────
platforms = sorted(vgsales['Platform'].dropna().unique())
pd.DataFrame({'platform_name': platforms}).to_sql(
    'dim_platform', engine, if_exists='append', index=False)
print(f'✅ dim_platform : {len(platforms)} rows')

# ── dim_developer ─────────────────────────────────────
devs = sorted(games['Primary_Developer'].dropna().unique())
pd.DataFrame({'developer_name': devs}).to_sql(
    'dim_developer', engine, if_exists='append', index=False)
print(f'✅ dim_developer: {len(devs)} rows')

# ── dim_publisher ─────────────────────────────────────
pubs = sorted(vgsales['Publisher'].dropna().unique())
pd.DataFrame({'publisher_name': pubs}).to_sql(
    'dim_publisher', engine, if_exists='append', index=False)
print(f'✅ dim_publisher: {len(pubs)} rows')

✅ dim_genre    : 24 rows
✅ dim_platform : 31 rows
✅ dim_developer: 466 rows
✅ dim_publisher: 578 rows


In [17]:
# FK maps fetch karo
with engine.connect() as conn:
    genre_map = dict(conn.execute(text('SELECT genre_name,  genre_id     FROM dim_genre')).fetchall())
    plat_map  = dict(conn.execute(text('SELECT platform_name, platform_id FROM dim_platform')).fetchall())
    dev_map   = dict(conn.execute(text('SELECT developer_name, developer_id FROM dim_developer')).fetchall())
    pub_map   = dict(conn.execute(text('SELECT publisher_name, publisher_id FROM dim_publisher')).fetchall())

print(f'genre_map={len(genre_map)} | plat_map={len(plat_map)} | dev_map={len(dev_map)} | pub_map={len(pub_map)}')

genre_map=24 | plat_map=31 | dev_map=466 | pub_map=578


## Step 4 — game_engagement Table

In [18]:
eng = games.copy()

# FK IDs
eng['genre_id']     = eng['Primary_Genre'].map(genre_map)
eng['developer_id'] = eng['Primary_Developer'].map(dev_map)

# Exact rename: CSV column → table column
eng = eng.rename(columns={
    'Game_Title'        : 'game_title',
    'Release Date'      : 'release_date',
    'Release Year'      : 'release_year',    # space wala
    'Release Month'     : 'release_month',   # space wala
    'Rating'            : 'rating',
    'Times Listed'      : 'times_listed',
    'Number of Reviews' : 'num_reviews',
    'Plays'             : 'plays',
    'Playing'           : 'playing',
    'Backlogs'          : 'backlogs',
    'Wishlist'          : 'wishlist',
    'Genres_Clean'      : 'genres_clean',
    'Primary_Genre'     : 'primary_genre',
    'Team_Clean'        : 'team_clean',
    'Primary_Developer' : 'primary_developer',
})

# Sirf table ke columns rakhna — extra columns drop
TABLE_COLS = ['game_id','game_title','release_date','release_year',
              'release_month','rating','times_listed','num_reviews',
              'plays','playing','backlogs','wishlist','genres_clean',
              'primary_genre','team_clean','primary_developer',
              'genre_id','developer_id']
eng = eng[[c for c in TABLE_COLS if c in eng.columns]]

# NaN → None (PostgreSQL ke liye)
for col in ['release_year','release_month','times_listed','num_reviews',
            'plays','playing','backlogs','wishlist']:
    if col in eng.columns:
        eng[col] = pd.to_numeric(eng[col], errors='coerce')
        eng[col] = eng[col].where(eng[col].notna(), None)

# release_date → string to date
eng['release_date'] = pd.to_datetime(eng['release_date'], errors='coerce')
eng['release_date'] = eng['release_date'].where(eng['release_date'].notna(), None)

eng.to_sql('game_engagement', engine, if_exists='append',
            index=False, method='multi', chunksize=100)

print(f'✅ game_engagement : {len(eng)} rows inserted')

✅ game_engagement : 1099 rows inserted


## Step 5 — game_sales Table

In [19]:
sal = vgsales.copy()

# FK IDs
sal['platform_id']  = sal['Platform'].map(plat_map)
sal['publisher_id'] = sal['Publisher'].map(pub_map)
sal['genre_id']     = sal['Genre'].map(genre_map)

sal = sal.rename(columns={
    'sales_id'    : 'sales_id',
    'Game_Title'  : 'game_title',
    'Platform'    : 'platform',
    'Year'        : 'release_year',
    'Genre'       : 'genre',
    'Publisher'   : 'publisher',
    'NA_Sales'    : 'na_sales',
    'EU_Sales'    : 'eu_sales',
    'JP_Sales'    : 'jp_sales',
    'Other_Sales' : 'other_sales',
    'Global_Sales': 'global_sales',
})

# Sirf table ke columns rakhna
TABLE_COLS_S = ['sales_id','game_title','platform','release_year','genre',
                'publisher','na_sales','eu_sales','jp_sales',
                'other_sales','global_sales',
                'platform_id','publisher_id','genre_id']
sal = sal[[c for c in TABLE_COLS_S if c in sal.columns]]

sal['release_year'] = pd.to_numeric(sal['release_year'], errors='coerce')
sal['release_year'] = sal['release_year'].where(sal['release_year'].notna(), None)

sal.to_sql('game_sales', engine, if_exists='append',
            index=False, method='multi', chunksize=500)

print(f'✅ game_sales : {len(sal)} rows inserted')

✅ game_sales : 16593 rows inserted


## Step 6 — merged_data Table

In [20]:
mer = merged.copy()

# Tiers add karo
def rating_tier(r):
    if pd.isna(r): return None
    if r >= 4.0:   return 'High (4+)'
    if r >= 3.0:   return 'Medium (3-4)'
    return 'Low (<3)'

def sales_tier(v):
    if pd.isna(v) or v == 0: return 'No Sales Data'
    if v >= 5:   return 'Blockbuster (5M+)'
    if v >= 1:   return 'Hit (1-5M)'
    return 'Mid-Tier (<1M)'

mer['rating_tier'] = mer['Rating'].apply(rating_tier)
mer['sales_tier']  = mer['Global_Sales'].apply(sales_tier)

mer = mer.rename(columns={
    'Game_Title'        : 'game_title',
    'Release Year'      : 'release_year',    # space wala
    'Rating'            : 'rating',
    'Plays'             : 'plays',
    'Backlogs'          : 'backlogs',
    'Wishlist'          : 'wishlist',
    'Primary_Genre'     : 'primary_genre',
    'Primary_Developer' : 'primary_developer',
    'Publisher'         : 'publisher',
    'NA_Sales'          : 'na_sales',
    'EU_Sales'          : 'eu_sales',
    'JP_Sales'          : 'jp_sales',
    'Other_Sales'       : 'other_sales',
    'Global_Sales'      : 'global_sales',
    'Sales_Genre'       : 'sales_genre',
    'Platforms'         : 'platforms',
})

TABLE_COLS_M = ['game_title','release_year','rating','rating_tier',
                'plays','backlogs','wishlist','primary_genre',
                'primary_developer','publisher','na_sales','eu_sales',
                'jp_sales','other_sales','global_sales',
                'sales_genre','platforms','sales_tier']
mer = mer[[c for c in TABLE_COLS_M if c in mer.columns]]

for col in ['release_year','plays','backlogs','wishlist']:
    if col in mer.columns:
        mer[col] = pd.to_numeric(mer[col], errors='coerce')
        mer[col] = mer[col].where(mer[col].notna(), None)

mer.to_sql('merged_data', engine, if_exists='append',
            index=False, method='multi', chunksize=100)

print(f'✅ merged_data : {len(mer)} rows inserted')

✅ merged_data : 1099 rows inserted


## Step 7 — Verify Row Counts

In [21]:
print('=' * 48)
print('  TABLE          EXPECTED    ACTUAL   STATUS')
print('=' * 48)

expected = {
    'dim_genre'      : '~30',
    'dim_platform'   : '31',
    'dim_developer'  : '~465',
    'dim_publisher'  : '~578',
    'game_engagement': '1099',
    'game_sales'     : '16593',
    'merged_data'    : '1099',
}

with engine.connect() as conn:
    for tbl, exp in expected.items():
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).scalar()
        status = '✅' if n > 0 else '❌ EMPTY'
        print(f'  {tbl:<20} {exp:>6}   {n:>6,}   {status}')

  TABLE          EXPECTED    ACTUAL   STATUS
  dim_genre               ~30       24   ✅
  dim_platform             31       31   ✅
  dim_developer          ~465      466   ✅
  dim_publisher          ~578      578   ✅
  game_engagement        1099    1,099   ✅
  game_sales            16593   16,593   ✅
  merged_data            1099    1,099   ✅


In [22]:
# Quick test queries
with engine.connect() as conn:
    print('── Top 5 Platforms by Sales ──')
    display(pd.read_sql('SELECT * FROM vw_platform_sales LIMIT 5', conn))

    print('\n── Top 5 Genres by Rating ──')
    display(pd.read_sql('SELECT * FROM vw_genre_rating LIMIT 5', conn))

    print('\n── Top 10 Best Selling Games ──')
    display(pd.read_sql(
        'SELECT game_title, platform, global_sales, genre FROM game_sales ORDER BY global_sales DESC LIMIT 10',
        conn
    ))

── Top 5 Platforms by Sales ──


,platform,total_games,total_global_sales_m,total_na_sales_m,total_eu_sales_m,total_jp_sales_m
0,PS2,2161,1255.64,583.84,339.29,139.20
1,Xbox 360,1264,978.67,600.05,280.41,12.41
2,PS3,1327,957.35,392.26,343.22,79.99
3,Wii,1324,926.69,507.71,268.38,69.33
4,Nintendo DS,2163,822.49,390.71,194.65,175.57



── Top 5 Genres by Rating ──


,genre,total_games,avg_rating,total_plays_m,total_wishlist_k
0,Point-and-Click,3,3.90,0.008,1.05
1,Strategy,1,3.80,0.004,0.07
2,RPG,77,3.71,0.272,45.23
3,Adventure,713,3.70,3.728,522.54
4,Music,6,3.70,0.017,1.10



── Top 10 Best Selling Games ──


,game_title,platform,global_sales,genre
0,Wii Sports,Wii,82.74,Sports
1,Super Mario Bros.,NES,40.24,Platform
2,Mario Kart Wii,Wii,35.82,Racing
3,Wii Sports Resort,Wii,33.00,Sports
4,Pokemon Red/Pokemon Blue,Game Boy,31.37,Role-Playing
5,Tetris,Game Boy,30.26,Puzzle
6,New Super Mario Bros.,Nintendo DS,30.01,Platform
7,Wii Play,Wii,29.02,Misc
8,New Super Mario Bros. Wii,Wii,28.62,Platform
9,Duck Hunt,NES,28.31,Shooter


## ✅ Database Setup Complete!

**pgAdmin4 mein verify karo:**
```
videogame_analytics → Schemas → public
  ├── Tables (7)  : dim_genre, dim_platform, dim_developer,
  │                 dim_publisher, game_engagement,
  │                 game_sales, merged_data
  └── Views  (6)  : vw_genre_rating, vw_platform_sales,
                    vw_publisher_performance, vw_yearly_trends,
                    vw_engagement_sales, vw_developer_stats
```
**Agle step:** EDA Notebook 🎮
